# SpikeDistill-RUL final experiments

Run in a Colab GPU runtime. All four subsets and all three seeds use the original baseline data preparation and the validation-selected attention BiLSTM checkpoints. Student configurations are fixed before official test scoring. Drive saves a checkpoint after each epoch.


## Teacher and student comparison on the official test set

The frozen attention BiLSTM is paired with each student on the same subset and seed. RMSE and MAE are in cycles. MSE is in cycles squared. The NASA score is summed over the engines in each subset. A positive student-minus-teacher RMSE means the student had more error. The primary table uses raw RUL. Capped RUL is shown separately.

Source: the verified per-seed test predictions summarized in [test_metrics_per_seed.csv](https://github.com/abhav-arora/ML3103-project/blob/344847b/results/test_metrics_per_seed.csv). The rows are included below so this section runs without retraining or a Drive mount.

In [1]:
import pandas as pd
from io import StringIO
from IPython.display import display

comparison_metrics = ["RMSE", "MAE", "MSE", "RAE_pct", "R2", "cmapss_score", "bias_cycles"]
comparison_models = ["attention_bilstm", "learnable_snn", "spike_output_kd", "spike_temporal_kd", "spikedistill"]

paired_test = pd.read_csv(StringIO('subset,seed,model,target,engines,RMSE,MAE,MSE,RAE_pct,R2,cmapss_score,bias_cycles\nFD001,42,attention_bilstm,raw,100,15.729126683262827,12.167774021402002,247.40542621813063,33.094100234453535,0.8567318422779979,490.8462955601786,2.8525064899772405\nFD001,42,attention_bilstm,capped,100,14.588087571579404,11.124055497422814,212.81229899606947,31.032040330914203,0.8674782950672646,457.9519538423725,3.9225064899772404\nFD001,43,attention_bilstm,raw,100,15.723856741213224,11.959728308096528,247.23967081819654,32.52825428125212,0.8568278283327261,493.25751794948917,3.4773240811377764\nFD001,43,attention_bilstm,capped,100,14.849758711023773,11.102661729231476,220.51533377562643,30.972359553746415,0.8626814891168627,469.684336776971,4.547324081137776\nFD001,44,attention_bilstm,raw,100,15.721141006339328,11.521373661905526,247.15427454120393,31.336010525428986,0.8568772798240215,386.54865433242315,0.33820190608501305\nFD001,44,attention_bilstm,capped,100,14.56507557858245,10.501214171797038,212.1414266098189,29.294541166058632,0.8678960582925933,352.01626914455034,1.4082019060850133\nFD002,42,attention_bilstm,raw,259,29.69830073866354,19.919453038801674,881.9890667641034,43.98644901612273,0.6950412625527989,15246.375239593934,-11.994675433788176\nFD002,42,attention_bilstm,capped,259,17.212604467494508,12.436457812225152,296.27375255441194,32.58277514631981,0.8393118564551231,1425.7569039643677,-4.500466939579681\nFD002,43,attention_bilstm,raw,259,28.410014596465672,18.567790728680876,807.1289293713925,41.001687076358145,0.7209250901927067,13871.749575826845,-11.852165077298764\nFD002,43,attention_bilstm,capped,259,14.755071971555504,11.111902843053276,217.7121488857828,29.112520401673518,0.8819208224488352,758.6070233856944,-4.35795658309027\nFD002,44,attention_bilstm,raw,259,29.03528242691586,19.330824201764777,843.0476256107689,42.68662957432842,0.7085057522794858,16964.496512202873,-11.65471054229914\nFD002,44,attention_bilstm,capped,259,16.26968967294065,11.836615707556282,264.7028020537917,31.01122473262335,0.856434795568537,1307.2470379015235,-4.160502048090647\nFD003,42,attention_bilstm,raw,100,14.166413947437844,10.566702886521817,200.68728413016146,29.5212073849007,0.8828854077934274,402.67274619766516,1.2652009877562527\nFD003,42,attention_bilstm,capped,100,12.989345560680366,9.10477368682623,168.72309809476675,26.57179871712728,0.8900097562431181,373.7686647235268,2.8252009877562525\nFD003,43,attention_bilstm,raw,100,15.146123027834667,10.969206844717265,229.40504277430358,30.645721147683563,0.8661266549542882,470.90415579237293,0.2543703991174694\nFD003,43,attention_bilstm,capped,100,13.621385333791311,9.44480601683259,185.54213841162502,27.56416502309247,0.879045458141094,428.7612477424385,1.8143703991174691\nFD003,44,attention_bilstm,raw,100,14.7743199486689,10.93784569080919,218.28052994563583,30.558104495801455,0.8726185599550117,388.3562424986465,1.151491851843894\nFD003,44,attention_bilstm,capped,100,13.192411067857895,9.398498402200639,174.0397097833395,27.429018707830306,0.8865438679196453,347.0368034645661,2.7114918518438937\nFD004,42,attention_bilstm,raw,248,27.69465258685941,19.77022028427511,766.9937819068385,42.54320029487824,0.7419999922315454,7011.774930023146,-7.1005127192174475\nFD004,42,attention_bilstm,capped,248,16.175482521311974,11.22722083596032,261.6462347972692,29.2699206022742,0.8583689672305366,1872.8774522485674,1.5930356678793272\nFD004,43,attention_bilstm,raw,248,26.646810416273237,19.06300506906043,710.0525053608078,41.02135591885,0.761153797826551,5913.236821664561,-6.931779745275215\nFD004,43,attention_bilstm,capped,248,14.050022353555898,10.428888881849426,197.4031281354204,27.188629670840804,0.8931442337345108,990.5039109109341,1.7617686418215595\nFD004,44,attention_bilstm,raw,248,26.9704449254232,19.595684313545785,727.4048994752861,42.16761931223716,0.7553168302761668,5509.870010102321,-7.026905149461761\nFD004,44,attention_bilstm,capped,248,14.463796699404332,10.911570524556502,209.20141496169967,28.44700461194465,0.8867577342329567,1007.1414741428616,1.6666432376350124\nFD001,42,learnable_snn,raw,100,18.623709191804288,14.26677038386464,346.8425440608955,38.80298304974173,0.7991495454776113,739.9045562904524,1.9477934125065806\nFD001,42,learnable_snn,capped,100,17.254187087297513,13.19677038386464,297.7069720434642,36.81415567234257,0.8146129913934592,681.0567214164168,3.01779341250658\nFD001,42,spike_output_kd,raw,100,17.75127411870189,13.093011967986822,315.10773283729554,35.610576731398695,0.8175266199385897,839.4102314205905,1.7353550638258453\nFD001,42,spike_output_kd,capped,100,16.504942676683857,12.023011967986822,272.4131327606201,33.53979961499379,0.8303638794853124,776.4632769715528,2.8053550638258455\nFD001,42,spike_temporal_kd,raw,100,18.194456540052617,13.079197446033358,331.0382487878635,35.57300378063425,0.808301536614077,759.7079939118721,0.23415185652673245\nFD001,42,spike_temporal_kd,capped,100,16.71178787167882,12.009197446033358,279.28385386799124,33.50126215871163,0.8260853688937654,693.6542256592322,1.3041518565267325\nFD001,42,spikedistill,raw,100,17.547649847599473,13.530073109939696,307.92001517395784,36.79930239436154,0.8216889015974582,536.6178197673669,0.9274852662533529\nFD001,42,spikedistill,capped,100,15.890043194146683,12.460073109939696,252.49347271184735,34.75904011476469,0.8427681781268708,452.11210567646623,1.9974852662533527\nFD001,43,learnable_snn,raw,100,18.768637671747168,14.26624915279448,352.261760053327,38.801565397404424,0.7960113722232837,812.3268119462376,0.94666762419045\nFD001,43,learnable_snn,capped,100,17.275153994727546,13.196249152794481,298.43094554155107,36.81270162857277,0.8141621612358734,737.5195293156477,2.0166676241904504\nFD001,43,spike_output_kd,raw,100,17.335256562479525,13.207788239121436,300.5111200869894,35.922747011253065,0.8259792632362111,581.2223972797058,3.6463135215640072\nFD001,43,spike_output_kd,capped,100,16.163563360785766,12.17305809557438,261.260780518136,33.95837335223137,0.8373086319275183,536.0225030188261,4.716313521564007\nFD001,43,spike_temporal_kd,raw,100,16.374371812760067,12.134575760513544,268.12005226251136,33.00380709032383,0.8447363644235144,421.0508738018996,1.4472667235136014\nFD001,43,spike_temporal_kd,capped,100,15.089503325102932,11.124575760513544,227.69311059829244,31.033491674375945,0.8582117699011329,379.8677753048136,2.5172667235136017\nFD001,43,spikedistill,raw,100,17.761896905115652,13.941948217824102,315.48498166795696,37.91952669179078,0.8173081617349932,720.7848919369661,3.547667024210096\nFD001,43,spikedistill,capped,100,16.662132418204816,12.902664875462651,277.6266567217919,35.993709028545354,0.8271173327053497,674.0230999293155,4.617667024210096\nFD001,44,learnable_snn,raw,100,17.472459763999105,14.091625650897623,305.2868502045676,38.32662169242592,0.8232137214039974,534.1931304019154,0.2771198949962853\nFD001,44,learnable_snn,capped,100,16.030618060860704,13.021625650897622,256.98071541319337,36.32556601918605,0.839973898585535,479.85488752453944,1.3471198949962855\nFD001,44,spike_output_kd,raw,100,17.036954425225964,14.103692220374942,290.2578160872266,38.35944053497395,0.8319167723566234,454.0202431331048,4.544115384593606\nFD001,44,spike_output_kd,capped,100,15.897854048993645,13.093692220374942,252.74176336310364,36.52660535156343,0.8426135634707698,409.2467613433,5.614115384593606\nFD001,44,spike_temporal_kd,raw,100,16.64173841055315,12.88471590846777,276.9474573252801,35.04404988268829,0.8396245684530667,402.4317265511313,0.006969954371452119\nFD001,44,spike_temporal_kd,capped,100,15.3603874343918,11.814715908467772,235.94150213462152,32.95872990338877,0.8530753613641091,361.81954208219156,1.0769699543714522\nFD001,44,spikedistill,raw,100,17.691856529033963,13.465374603271485,313.0017874439216,36.62333439389315,0.818746136104358,507.85690660486915,0.2224692809581768\nFD001,44,spikedistill,capped,100,16.45125517551811,12.395374603271485,270.6437968500114,34.57855497885872,0.8314656739425815,459.43075212165945,1.2924692809581768\nFD002,42,learnable_snn,raw,259,28.840173233336355,19.961500726467623,831.7555921288507,44.07929938034515,0.7124101139134585,30611.802516394517,-7.622231992716725\nFD002,42,learnable_snn,capped,259,17.18850118542677,12.614304902326877,295.4445730014175,33.048723878242136,0.8397615733871404,1889.3708748821214,-0.12802349850823097\nFD002,42,spike_output_kd,raw,259,28.785816318045054,19.589626977802705,828.6232210962288,43.258121928623915,0.7134931703268779,11221.152800807853,-10.229309126332\nFD002,42,spike_output_kd,capped,259,16.39647204218674,12.137210544932303,268.8442954302114,31.798765215954518,0.8541885996891432,1267.3084717552683,-2.735100632123505\nFD002,42,spike_temporal_kd,raw,259,29.638435676872888,19.830354714715803,878.4368693721317,43.78970069767287,0.696269479174342,31915.643510809154,-11.420633231235747\nFD002,42,spike_temporal_kd,capped,259,16.591011255480566,12.367379169436495,275.26165447948284,32.40179323656869,0.8507080567682883,1206.4688792796073,-3.926424737027253\nFD002,42,spikedistill,raw,259,28.961593889419426,19.491279003242727,838.7739206156567,43.04094838674028,0.7099834391677289,11412.51504409559,-9.874966596947214\nFD002,42,spikedistill,capped,259,16.224089574621832,12.128903697816561,263.22108252535287,31.77700177367588,0.857238426528844,1333.8975716389778,-2.380758102738719\nFD002,43,learnable_snn,raw,259,29.99353447910684,20.451911370971143,899.612110549371,45.16223186696506,0.6889478750208542,16612.300481445644,-12.9448370819185\nFD002,43,learnable_snn,capped,259,16.81790599579444,12.957919892901735,282.8419620833786,33.948974588199086,0.8465967727079651,1193.9664986713701,-5.450628587710006\nFD002,43,spike_output_kd,raw,259,29.89985855781327,19.686378903197966,894.0015417772396,43.47177104967761,0.6908877992598101,15936.268504265954,-11.99128192266212\nFD002,43,spike_output_kd,capped,259,16.036575740571042,12.20209550287963,257.1717614830717,31.96875992241329,0.860519358975982,902.9741192042591,-4.497073428453626\nFD002,43,spike_temporal_kd,raw,259,29.626405938281614,19.683409457875264,877.7239288198481,43.46521387388375,0.6965159872761907,16869.235854641192,-11.478013917257197\nFD002,43,spike_temporal_kd,capped,259,16.550945365249937,12.24348157888789,273.93379248348833,32.07718896459925,0.8514282409802899,1332.6324275262014,-3.9838054230487026\nFD002,43,spikedistill,raw,259,29.069575611042886,19.677648080908423,845.0402262061394,43.45249151078869,0.7078167857325078,14194.892452346638,-10.842408151556397\nFD002,43,spikedistill,capped,259,16.188893619470065,12.289643557001495,262.0802766225186,32.198130584463584,0.8578571582966463,1122.8147808502436,-3.348199657347902\nFD002,44,learnable_snn,raw,259,29.47253464372323,19.805197076318233,868.6302983254659,43.734147205486856,0.6996602236152561,16267.376518113786,-11.769313177331188\nFD002,44,learnable_snn,capped,259,16.529857209892487,12.310988582109738,273.23617937943465,32.254053272747925,0.8518066010396488,1077.4618316356837,-4.275104683122695\nFD002,44,spike_output_kd,raw,259,28.492650931769386,19.342002514965515,811.8311571196589,42.71131370108294,0.7192992362095483,11695.354757585797,-10.056654272708883\nFD002,44,spike_output_kd,capped,259,15.859999088854332,11.861443369343704,251.53957109846027,31.076271720571416,0.8635740548752696,1355.2746123291379,-2.5624457785003893\nFD002,44,spike_temporal_kd,raw,259,29.682897927145618,19.910242047204015,881.0744293533456,43.96610915982702,0.6953575098630178,15888.339974516226,-11.05121530782302\nFD002,44,spike_temporal_kd,capped,259,16.167068375729016,12.41603355299552,261.37409986549727,32.529264809529614,0.8582401629709449,1168.5957171720897,-3.557006813614525\nFD002,44,spikedistill,raw,259,28.97683069160189,19.30912163424302,839.6567169297613,42.63870562390894,0.7096782013144298,20157.64441535619,-9.834096450567188\nFD002,44,spikedistill,capped,259,15.505467964214219,11.814913140034527,240.41953678927342,30.95436530461451,0.86960515838644,1081.6634906518973,-2.3398879563586936\nFD003,42,learnable_snn,raw,100,21.888892544978003,16.779904664605855,479.1236168455935,46.87962279459416,0.7203989916619903,2460.802404326374,5.850751899480819\nFD003,42,learnable_snn,capped,100,20.793395225111116,15.241714004129172,432.3652849876737,44.482133280010885,0.7181419519626342,2414.4960763825666,7.41075189948082\nFD003,42,spike_output_kd,raw,100,21.09212494380217,16.067806453853844,444.8777346449617,44.89016599015982,0.7403837781723307,2184.418839718614,3.5131341929733755\nFD003,42,spike_output_kd,capped,100,19.7899492102773,14.507806453853846,391.64208974535507,42.340263050868074,0.7446893199391629,2120.592408975465,5.0731341929733755\nFD003,42,spike_temporal_kd,raw,100,19.630996508970412,14.708552976176144,385.3760239352085,41.092689688034014,0.7751070473399306,1205.833358644812,0.4557271174341428\nFD003,42,spike_temporal_kd,capped,100,17.98015505600095,13.194900355860591,323.28597583783653,38.50861629386598,0.7892505312721734,1132.4568586396988,2.015727117434143\nFD003,42,spikedistill,raw,100,20.401892336557804,15.885746795088053,416.2372109124961,44.38152852769225,0.7570974592211753,1419.168119536948,1.3518840785324568\nFD003,42,spikedistill,capped,100,18.90967689358356,14.35416479960084,357.575880219728,41.89186803833916,0.766897012495236,1355.4973187006444,2.9118840785324567\nFD003,43,learnable_snn,raw,100,19.582041214223253,14.899156812950968,383.4563381155381,41.62519783690651,0.7762273137430059,1185.5325120440605,4.194424437657\nFD003,43,learnable_snn,capped,100,18.291054024893278,13.40194017343223,334.5626573415646,39.11285101162776,0.7818992855840037,1132.8800594583518,5.754424437657001\nFD003,43,spike_output_kd,raw,100,21.62018226159414,16.135944000184537,467.43228102455004,45.08052836312786,0.7272216761831658,2097.725289251516,3.9061266056448227\nFD003,43,spike_output_kd,capped,100,20.18887580804391,14.580617302358151,407.59070639262063,42.55275764737617,0.7342924492532505,2031.045009289975,5.466126605644822\nFD003,43,spike_temporal_kd,raw,100,17.512298574774235,13.06034484937787,306.68060137203975,36.4879331762602,0.8210311444343528,867.0061561185432,1.8090767937898626\nFD003,43,spike_temporal_kd,capped,100,15.963989046109472,11.501502967625857,254.8489462643032,33.56652590304293,0.8338644913629367,805.3801742074604,3.369076793789863\nFD003,43,spikedistill,raw,100,19.599968706939574,14.819078742861748,384.1587733130106,41.40147608193014,0.7758173953365652,1472.888523618729,2.7512352134287346\nFD003,43,spikedistill,capped,100,18.316708512902498,13.31040072619915,335.5018107466349,38.8456979938571,0.7812870533934191,1419.9968692645175,4.311235213428734\nFD003,44,learnable_snn,raw,100,18.548833468887008,13.965029410608112,344.0592230565028,39.015436867507354,0.7992181927329364,1008.9602363986028,1.7835185782238852\nFD003,44,learnable_snn,capped,100,17.038818167205125,12.52409535098821,290.32132453507944,36.55090749395358,0.8107401202679513,946.1802084444386,3.343518578223885\nFD003,44,spike_output_kd,raw,100,18.237290593283962,14.067710877358913,332.59876818388364,39.30230789124009,0.8059061426183816,812.5877518391582,2.5241038727760317\nFD003,44,spike_output_kd,capped,100,16.79570243517465,12.527431372106076,282.0956202909317,36.560643494507694,0.8161024400990966,758.8022834746299,4.084103872776032\nFD003,44,spike_temporal_kd,raw,100,20.080271559533923,14.686143423765898,403.21730590462687,41.03008198048226,0.7646954536440604,1374.6188386733033,1.72766208305955\nFD003,44,spike_temporal_kd,capped,100,18.665249906462844,13.139956204146147,348.3915540707112,38.348264703562094,0.7728842559923039,1307.7972457412561,3.28766208305955\nFD003,44,spikedistill,raw,100,20.550292746741437,15.591090321466327,422.3145319767737,43.5583185861895,0.7535509316908627,1426.685102412432,2.953112112656236\nFD003,44,spikedistill,capped,100,19.29532338214487,14.10152542822063,372.3095044215464,41.15455344324388,0.7572921929081152,1366.7080625468714,4.513112112656236\nFD004,42,learnable_snn,raw,248,30.288131140222394,22.605026852639934,917.3708879673094,48.64337226570375,0.6914164080526172,8106.569036025995,-9.296360561443914\nFD004,42,learnable_snn,capped,248,18.25242466536961,14.069337041087207,333.1510061649929,36.679458268091864,0.8196629081710514,1867.662671114275,-0.6028121743471394\nFD004,42,spike_output_kd,raw,248,29.485382302858266,22.19933238856855,869.3877695457074,47.77036525843029,0.7075568843088271,6574.490385164215,-9.024344824645068\nFD004,42,spike_output_kd,capped,248,18.12251956720492,13.58692164902365,328.4257154637252,35.421777455598296,0.8222207428086299,1864.8267134702244,-0.33079643754829274\nFD004,42,spike_temporal_kd,raw,248,28.08756177670053,20.34848863009604,788.9111265599685,43.78756609893633,0.7346274747168491,7321.587870553396,-6.591919701276047\nFD004,42,spike_temporal_kd,capped,248,16.553790912954884,11.847987053164792,274.0279935898277,30.888288865945647,0.8516666300589308,2364.199185371852,2.1016286858207276\nFD004,42,spikedistill,raw,248,28.492451684208852,21.472602509893477,811.8198029769759,46.20652761048373,0.7269214947819677,6260.663464277226,-7.15697311570928\nFD004,42,spikedistill,capped,248,17.148084601513137,12.918709591722056,294.056805500652,33.67971553765197,0.8408248867474961,2362.216245223846,1.5365752713874952\nFD004,43,learnable_snn,raw,248,28.305046402496114,21.17535087520317,801.1756518474582,45.566876880710716,0.73050195545699,6045.487727483788,-6.160378314013923\nFD004,43,learnable_snn,capped,248,17.64237087422012,12.952975522666689,311.2532500635304,33.76904851620174,0.8315163247294322,1775.0198184330252,2.5331700730828497\nFD004,43,spike_output_kd,raw,248,29.406148193838156,22.09015872197286,864.7215515979711,47.53543630477564,0.7091264984245664,6663.302959439538,-6.927216741586885\nFD004,43,spike_output_kd,capped,248,17.977629741821282,13.619526298656579,323.1951711340172,35.50677939151498,0.825052074954254,1757.322545631539,1.7663316455098892\nFD004,43,spike_temporal_kd,raw,248,29.468561324338125,20.99853377903421,868.3961065262768,45.186387183089195,0.7078904581561675,7752.361778599736,-9.675648559726053\nFD004,43,spike_temporal_kd,capped,248,16.99714250179984,12.41160728608168,288.90285322649055,32.357674719163484,0.8436147590496892,2026.8682212845479,-0.9821001726292797\nFD004,43,spikedistill,raw,248,27.4629304656229,20.867345890060307,754.2125497596385,44.90408619925453,0.7462993204282747,5129.807626433772,-5.625638649498503\nFD004,43,spikedistill,capped,248,16.492605004477934,12.36018794038964,272.0060198337306,32.22362193906617,0.8527611393360495,1627.3538387547428,3.0679097375982716\nFD004,44,learnable_snn,raw,248,29.21220284118122,21.342803426229604,853.3527948343161,45.92721517314099,0.712950700657404,7406.118780518865,-7.943444772315542\nFD004,44,learnable_snn,capped,248,17.564525851428044,12.728257855002377,308.5125683854841,33.18319843036436,0.8329998758948247,2032.0862732374592,0.7501036147812321\nFD004,44,spike_output_kd,raw,248,28.72502482516844,21.305523902988963,825.1270512065432,45.84699400204025,0.7224452262285648,6561.526142671464,-8.538478403953054\nFD004,44,spike_output_kd,capped,248,16.579343136087992,12.627658507366094,274.87461882414806,32.92093095019479,0.8512083455879221,1675.9034774108432,0.1550699831437204\nFD004,44,spike_temporal_kd,raw,248,27.8484408641813,20.66764572884647,775.5356585658029,44.474354823717086,0.7391266909136508,5406.058531413501,-7.607143526866792\nFD004,44,spike_temporal_kd,capped,248,16.50591572110955,12.28057417799268,272.4452537923714,32.01606491865525,0.852523378761211,1486.96521576652,1.0864048602299825\nFD004,44,spikedistill,raw,248,29.022172380255423,21.558520095873504,842.2864896692607,46.391412200369054,0.716673164758031,6880.904258287144,-7.065193266307395\nFD004,44,spikedistill,capped,248,17.618867834443552,12.997987268342365,310.4245037675896,33.88639636580287,0.8319649311994962,2177.5988415222982,1.6283551207893798\n'))

In [2]:
for target in ("raw", "capped"):
    rows = paired_test.loc[paired_test.target.eq(target)]
    assert not rows.duplicated(["subset", "seed", "model"]).any()
    assert rows.groupby(["subset", "seed"]).engines.nunique().eq(1).all()
    assert rows.groupby(["subset", "seed"]).model.nunique().eq(len(comparison_models)).all()
    paired_rmse = rows.pivot(index=["subset", "seed"], columns="model", values="RMSE")[comparison_models]
    paired_rmse["spikedistill_minus_teacher"] = paired_rmse.spikedistill - paired_rmse.attention_bilstm
    print(f"{target.upper()} test RMSE by subset and seed")
    display(paired_rmse.round(3))
    print(f"{target.upper()} teacher and student metrics across three seeds")
    aggregate = rows.groupby(["subset", "model"])[comparison_metrics].agg(["mean", "std"])
    display(aggregate.round(3))

RAW test RMSE by subset and seed
RAW teacher and student metrics across three seeds
CAPPED test RMSE by subset and seed
CAPPED teacher and student metrics across three seeds


model        attention_bilstm  ...  spikedistill_minus_teacher
subset seed                    ...                            
FD001  42              15.729  ...                       1.819
       43              15.724  ...                       2.038
       44              15.721  ...                       1.971
FD002  42              29.698  ...                      -0.737
       43              28.410  ...                       0.660
       44              29.035  ...                      -0.058
FD003  42              14.166  ...                       6.235
       43              15.146  ...                       4.454
       44              14.774  ...                       5.776
FD004  42              27.695  ...                       0.798
       43              26.647  ...                       0.816
       44              26.970  ...                       2.052

[12 rows x 6 columns]

RMSE         ... bias_cycles       
                            mean    std  ...        mean    std
subset model                             ...                   
FD001  attention_bilstm   15.725  0.004  ...       2.223  1.662
       learnable_snn      18.288  0.710  ...       1.057  0.841
       spike_output_kd    17.374  0.359  ...       3.309  1.435
       spike_temporal_kd  17.070  0.983  ...       0.563  0.774
       spikedistill       17.667  0.109  ...       1.566  1.752
FD002  attention_bilstm   29.048  0.644  ...     -11.834  0.171
       learnable_snn      29.435  0.578  ...     -10.779  2.796
       spike_output_kd    29.059  0.742  ...     -10.759  1.071
       spike_temporal_kd  29.649  0.030  ...     -11.317  0.232
       spikedistill       29.003  0.058  ...     -10.184  0.571
FD003  attention_bilstm   14.696  0.495  ...       0.890  0.554
       learnable_snn      20.007  1.710  ...       3.943  2.045
       spike_output_kd    20.317  1.820  ...       3.314  0.712
       spike_temporal_kd  19.075  1.371  ...       1.331  0.759
       spikedistill       20.184  0.511  ...       2.352  0.872
FD004  attention_bilstm   27.104  0.537  ...      -7.020  0.085
       learnable_snn      29.268  0.993  ...      -7.800  1.573
       spike_output_kd    29.206  0.418  ...      -8.163  1.098
       spike_temporal_kd  28.468  0.875  ...      -7.958  1.572
       spikedistill       28.326  0.793  ...      -6.616  0.859

[20 rows x 14 columns]

model        attention_bilstm  ...  spikedistill_minus_teacher
subset seed                    ...                            
FD001  42              14.588  ...                       1.302
       43              14.850  ...                       1.812
       44              14.565  ...                       1.886
FD002  42              17.213  ...                      -0.989
       43              14.755  ...                       1.434
       44              16.270  ...                      -0.764
FD003  42              12.989  ...                       5.920
       43              13.621  ...                       4.695
       44              13.192  ...                       6.103
FD004  42              16.175  ...                       0.973
       43              14.050  ...                       2.443
       44              14.464  ...                       3.155

[12 rows x 6 columns]

RMSE         ... bias_cycles       
                            mean    std  ...        mean    std
subset model                             ...                   
FD001  attention_bilstm   14.668  0.158  ...       3.293  1.662
       learnable_snn      16.853  0.713  ...       2.127  0.841
       spike_output_kd    16.189  0.304  ...       4.379  1.435
       spike_temporal_kd  15.721  0.869  ...       1.633  0.774
       spikedistill       16.334  0.399  ...       2.636  1.752
FD002  attention_bilstm   16.079  1.240  ...      -4.340  0.171
       learnable_snn      16.845  0.330  ...      -3.285  2.796
       spike_output_kd    16.098  0.273  ...      -3.265  1.071
       spike_temporal_kd  16.436  0.234  ...      -3.822  0.232
       spikedistill       15.973  0.405  ...      -2.690  0.571
FD003  attention_bilstm   13.268  0.323  ...       2.450  0.554
       learnable_snn      18.708  1.912  ...       5.503  2.045
       spike_output_kd    18.925  1.855  ...       4.874  0.712
       spike_temporal_kd  17.536  1.404  ...       2.891  0.759
       spikedistill       18.841  0.493  ...       3.912  0.872
FD004  attention_bilstm   14.896  1.127  ...       1.674  0.085
       learnable_snn      17.820  0.377  ...       0.893  1.573
       spike_output_kd    17.560  0.852  ...       0.530  1.098
       spike_temporal_kd  16.686  0.271  ...       0.735  1.572
       spikedistill       17.087  0.566  ...       2.078  0.859

[20 rows x 14 columns]

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import torch
from google.colab import drive

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU under Runtime > Change runtime type.")
project = Path("/content/drive/MyDrive/ML projec")
raw = project / "CMaps"
runs = project / "SpikeDistill-RUL" / "colab_runs"
baselines = runs / "baselines_v1"
output = runs / "spikedistill_v1"
data_root = Path("/content/spikedistill_baselines_data")
repo = Path("/content/spikedistill_baseline_source")
subsets = ["FD001", "FD002", "FD003", "FD004"]
seeds = ["42", "43", "44"]
epochs = "30"
revision = (baselines / "git_revision.txt").read_text().strip()
assert raw.is_dir(), raw
assert (baselines / "train_summary.json").is_file()
output.mkdir(parents=True, exist_ok=True)

def run(*command):
    with subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "https://github.com/abhav-arora/ML3103-project.git", str(repo)], check=True)
run("git", "checkout", "--detach", revision)
(output / "environment.json").write_text(json.dumps({
    "python": sys.version, "torch": str(torch.__version__), "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda, "baseline_revision": revision
}, indent=2))
run(sys.executable, "prepare_data.py", "--data-dir", str(raw), "--output-dir", str(data_root), "--subsets", *subsets)


## Distillation source

The next four cells write the Python module used by the training command. Run them in order. The first cell creates the file and the other three append to it.


### Imports and experiment settings


In [ ]:
%%writefile /content/spikedistill_baseline_source/distill.py
"""Train matched spiking students and compare their frozen checkpoints."""
import argparse
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

from baseline_models import HardSpike, Sequence
from metrics import regression_metrics
from run_baselines import aggregate, batches, predict_cycles, read_split, save_result, write_json
from train_pilot import digest, load_latest, restore_rng, rng_state, save_checkpoint

VARIANTS = {
    "learnable_snn": (1.0, 0.0, 0.0, 0.0, 0.0),
    "spike_output_kd": (0.5, 0.5, 0.0, 0.0, 0.0),
    "spike_temporal_kd": (0.5, 0.5, 0.1, 0.1, 0.0),
    "spikedistill": (0.5, 0.5, 0.1, 0.1, 0.01),
}




### Teacher and spiking student


In [ ]:
%%writefile -a /content/spikedistill_baseline_source/distill.py
class Teacher(Sequence):
    """Expose the same BiLSTM states and attention used by the baseline."""
    def __init__(self, features):
        super().__init__(features, "attention_bilstm")

    def signals(self, x, mask):
        order = torch.argsort(~mask, dim=1, stable=True)
        observed = (x * mask.unsqueeze(-1)).gather(1, order.unsqueeze(-1).expand_as(x))
        lengths = mask.sum(1)
        packed = pack_padded_sequence(observed, lengths.cpu(), batch_first=True, enforce_sorted=False)
        output, _ = self.cell(packed)
        states, _ = pad_packed_sequence(output, batch_first=True, total_length=x.size(1))
        valid = torch.arange(x.size(1), device=x.device)[None, :] < lengths[:, None]
        weights = self.attention(states).squeeze(-1).masked_fill(~valid, -torch.inf).softmax(1)
        prediction = self.readout((states * weights.unsqueeze(-1)).sum(1)).squeeze(-1)
        inverse = order.argsort(1)
        return prediction, states.gather(1, inverse.unsqueeze(-1).expand_as(states)), weights.gather(1, inverse)


class Student(nn.Module):
    def __init__(self, features, width=48):
        super().__init__()
        self.input = nn.Linear(features, width)
        self.recurrent = nn.Linear(width, width, bias=False)
        self.leak_logits = nn.Parameter(torch.full((width,), float(np.log(9))))
        self.threshold_logits = nn.Parameter(torch.full((width,), float(np.log(np.expm1(1 - 1e-4)))))
        self.attention = nn.Linear(width, 1, bias=False)
        self.readout = nn.Linear(width, 1)

    def signals(self, x, mask):
        currents = self.input(x)
        voltage = currents.new_zeros(currents.size(0), currents.size(2))
        spikes = torch.zeros_like(voltage)
        trace, events = [], []
        leak = self.leak_logits.sigmoid()
        threshold = F.softplus(self.threshold_logits) + 1e-4
        for current, valid in zip(currents.unbind(1), mask.unbind(1)):
            candidate = leak * voltage + current + self.recurrent(spikes)
            fired = HardSpike.apply(candidate - threshold)
            valid = valid.unsqueeze(-1)
            voltage = torch.where(valid, candidate - threshold * fired.detach(), voltage)
            spikes = torch.where(valid, fired, spikes)
            trace.append(torch.where(valid, voltage, 0))
            events.append(torch.where(valid, spikes, 0))
        membrane = torch.stack(trace, dim=1)
        events = torch.stack(events, dim=1)
        attention = self.attention(membrane).squeeze(-1).masked_fill(~mask, -torch.inf).softmax(1)
        prediction = self.readout((events * attention.unsqueeze(-1)).sum(1)).squeeze(-1)
        return prediction, membrane, attention, events

    def forward(self, x, mask):
        return self.signals(x, mask)[0]




### Checkpoint loading and training loop


In [ ]:
%%writefile -a /content/spikedistill_baseline_source/distill.py
def teacher_for(root, subset, seed, train_hash, validation_hash, cap, device):
    folder = root / subset / f"seed_{seed}" / "attention_bilstm"
    metrics = json.loads((folder / "validation_metrics.json").read_text())
    path = folder / f"epoch_{metrics['selected_epoch']:04d}.pt"
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    config = checkpoint["config"]
    sources = config["source_sha256"]
    if (config != metrics["config"] or checkpoint["epoch"] != metrics["selected_epoch"]
            or (config["subset"], config["seed"], config["cap"]) != (subset, seed, cap)
            or (sources["train.npz"], sources["validation.npz"]) != (train_hash, validation_hash)):
        raise ValueError(f"Teacher checkpoint or prepared data mismatch for {subset} seed {seed}")
    teacher = Teacher(config["features"]).to(device)
    teacher.load_state_dict(checkpoint["model"])
    return teacher.eval().requires_grad_(False), digest(path)


def train(model, adapter, teacher, config, directory, train_data, validation, device):
    directory.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.Adam([*model.parameters(), *adapter.parameters()], lr=config["lr"])
    generator = torch.Generator().manual_seed(config["seed"])
    loader = batches(train_data, config["cap"], config["batch_size"], generator)
    val_loader = batches(validation, config["cap"], config["batch_size"])
    history = []
    prior, _ = load_latest(directory)
    if prior is not None:
        if prior["config"] != config:
            raise ValueError(f"Run configuration changed: {directory}")
        model.load_state_dict(prior["model"])
        adapter.load_state_dict(prior["adapter"])
        optimizer.load_state_dict(prior["optimizer"])
        restore_rng(prior["rng"], generator)
        history = prior["history"]
    supervised, output, hidden, attention, sparse = config["loss_weights"]
    for epoch in range(len(history) + 1, config["epochs"] + 1):
        start = time.perf_counter()
        model.train()
        loss_sum = event_count = event_slots = 0
        for x, mask, target in loader:
            x, mask, target = x.to(device), mask.to(device), target.to(device)
            optimizer.zero_grad(set_to_none=True)
            pred, membrane, weights, spikes = model.signals(x, mask)
            rate = spikes.sum() / (mask.sum() * spikes.size(-1))
            loss = supervised * F.mse_loss(pred, target) + sparse * rate
            if output:
                with torch.no_grad():
                    reference, states, reference_weights = teacher.signals(x, mask)
                loss = loss + output * F.mse_loss(pred, reference)
                if hidden:
                    loss = loss + hidden * F.mse_loss(adapter(membrane[mask]), states[mask])
                    loss = loss + attention * F.mse_loss(weights[mask], reference_weights[mask])
            if not torch.isfinite(loss):
                raise RuntimeError(f"Nonfinite loss at {directory}, epoch {epoch}")
            loss.backward()
            nn.utils.clip_grad_norm_([*model.parameters(), *adapter.parameters()], 1.0, error_if_nonfinite=True)
            optimizer.step()
            loss_sum += loss.item() * len(x)
            event_count += spikes.detach().sum().item()
            event_slots += mask.sum().item() * spikes.size(-1)
        val = predict_cycles(model, val_loader, device, config["cap"])
        history.append(dict(epoch=epoch, loss=loss_sum / len(train_data["y"]),
                            spike_rate=event_count / event_slots, seconds=time.perf_counter() - start,
                            validation=regression_metrics(validation["y"], val)))
        save_checkpoint(dict(epoch=epoch, config=config, model=model.state_dict(), adapter=adapter.state_dict(),
                             optimizer=optimizer.state_dict(), rng=rng_state(generator), history=history), directory)
        write_json(directory / "history.json", history)
        print(f'{directory}: epoch {epoch}/{config["epochs"]}, RMSE {history[-1]["validation"]["RMSE"]:.3f} cycles', flush=True)




### Experiment runner


In [ ]:
%%writefile -a /content/spikedistill_baseline_source/distill.py
def run(args):
    device = torch.device(args.device)
    results = []
    for subset in args.subsets:
        prepared = args.data_root / subset
        data = {split: read_split(prepared, split) for split in ("train", "validation")}
        manifest = json.loads((prepared / "split_manifest.json").read_text())
        cap = float(manifest["rul_cap"])
        train_hash, validation_hash = (digest(prepared / f"{split}.npz") for split in data)
        for seed in args.seeds:
            teacher, teacher_hash = teacher_for(args.baselines, subset, seed, train_hash, validation_hash, cap, device)
            for name in VARIANTS:
                directory = args.output / subset / f"seed_{seed}" / name
                config = dict(model=name, subset=subset, seed=seed, epochs=args.epochs, batch_size=128, lr=0.001,
                              cap=cap, loss_weights=list(VARIANTS[name]), teacher_sha256=teacher_hash,
                              train_sha256=train_hash, validation_sha256=validation_hash, code_sha256=digest(__file__))
                random.seed(seed)
                np.random.seed(seed)
                torch.manual_seed(seed)
                model = Student(data["train"]["X"].shape[-1]).to(device)
                adapter = nn.Linear(48, teacher.readout.in_features).to(device)
                if args.phase == "train":
                    train(model, adapter, teacher, config, directory, data["train"], data["validation"], device)
                latest, _ = load_latest(directory)
                if latest is None or latest["config"] != config or latest["epoch"] != args.epochs:
                    raise RuntimeError(f"Complete the matching training run first: {directory}")
                best = min(latest["history"], key=lambda row: row["validation"]["RMSE"])["epoch"]
                state = torch.load(directory / f"epoch_{best:04d}.pt", map_location="cpu", weights_only=True)
                if state["config"] != config or state["epoch"] != best:
                    raise RuntimeError(f"Selected checkpoint mismatch: {directory}")
                model.load_state_dict(state["model"])
                split = "validation" if args.phase == "train" else "test"
                if split not in data:
                    data[split] = read_split(prepared, split)
                    if len(np.unique(data[split]["engine_id"])) != len(data[split]["engine_id"]):
                        raise ValueError("Official test requires one prediction per engine")
                prediction = predict_cycles(model, batches(data[split], cap, 128), device, cap)
                results.append(save_result(directory, config, split, data[split], prediction,
                                           best, sum(p.numel() for p in model.parameters())))
    write_json(args.output / f"{args.phase}_summary.json", aggregate(results))


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    for path in ("data-root", "baselines", "output"):
        parser.add_argument(f"--{path}", type=Path, required=True)
    parser.add_argument("--subsets", nargs="+", choices=["FD001", "FD002", "FD003", "FD004"],
                        default=["FD001", "FD002", "FD003", "FD004"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[42, 43, 44])
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--phase", choices=["train", "test"], default="train")
    parser.add_argument("--device", choices=["cpu", "cuda"], default="cuda")
    args = parser.parse_args()
    if args.epochs < 1 or args.output.resolve() == args.baselines.resolve():
        parser.error("Use positive epochs and a separate student output directory")
    run(args)


In [ ]:
import hashlib
assert hashlib.sha256((repo / "distill.py").read_bytes()).hexdigest() == "69ae00ea000e2994b6081021a172a1e35fdba21776cbf46580e43f95bb481ae6", "Run the four source cells in order before training."


The same student architecture is used in all four variants. The comparison adds output matching, then hidden-state and attention matching, then a firing-rate penalty. The teacher stays frozen. Checkpoint selection uses capped validation RMSE.


In [ ]:
common = ["--data-root", str(data_root), "--subsets", *subsets, "--seeds", *seeds,
          "--epochs", epochs, "--device", "cuda"]
student_command = [sys.executable, "distill.py", *common, "--baselines", str(baselines), "--output", str(output)]
run(*student_command, "--phase", "train")


The next cell checks that every planned validation result exists, then evaluates the frozen models on the official test trajectories. The comparison uses raw official RUL in cycles. Capped metrics remain in the JSON summaries for reference.


In [ ]:
def complete_validation(folder, models_per_subset):
    summary = json.loads((folder / "train_summary.json").read_text())
    for subset in subsets:
        entries = [r for r in summary if r["subset"] == subset and r["target"] == "capped"]
        assert len({r["model"] for r in entries}) == models_per_subset, f"Incomplete validation: {folder}, {subset}"
        assert all(r["seeds"] == list(map(int, seeds)) for r in entries)

complete_validation(baselines, 12)
complete_validation(output, 4)
run(sys.executable, "run_baselines.py", *common, "--output", str(baselines), "--phase", "test")
run(*student_command, "--phase", "test")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def table(folder, filename, target):
    rows = json.loads((folder / filename).read_text())
    return pd.DataFrame([
        {"subset": row["subset"], "model": row["model"], **{
            f"{metric}_{stat}": row[metric][stat] if row[metric] is not None else None
            for metric in ("MAE", "MSE", "RMSE", "R2", "cmapss_score")
            for stat in ("mean", "sample_sd")
        }} for row in rows if row["target"] == target
    ])

validation = pd.concat([table(folder, "train_summary.json", "capped") for folder in (baselines, output)])
test = pd.concat([table(folder, "test_summary.json", "raw") for folder in (baselines, output)])
comparison = validation.merge(test, on=["subset", "model"], suffixes=("_validation_capped", "_test_raw"))
comparison.to_csv(output / "final_comparison.csv", index=False)
display(comparison.sort_values(["subset", "RMSE_mean_test_raw"]))

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for subset, ax in zip(subsets, axes.flat):
    subset_rows = test[test["subset"] == subset].sort_values("RMSE_mean")
    ax.barh(subset_rows.model, subset_rows.RMSE_mean, xerr=subset_rows.RMSE_sample_sd.fillna(0))
    ax.invert_yaxis()
    ax.set(title=subset, xlabel="Official test RMSE (cycles), mean ± sample SD")
fig.savefig(output / "test_rmse.png", dpi=160)

errors = []
for subset in subsets:
    for seed in seeds:
        file = output / subset / f"seed_{seed}" / "spikedistill" / "test_predictions.csv"
        predictions = pd.read_csv(file)
        predictions["error"] = predictions.prediction_cycles - predictions.target_raw_cycles
        predictions["subset"], predictions["seed"] = subset, seed
        errors.append(predictions)
residuals = pd.concat(errors, ignore_index=True)
residuals.to_csv(output / "spikedistill_test_residuals.csv", index=False)
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
for subset, ax in zip(subsets, axes.flat):
    rows = residuals[residuals.subset == subset]
    ax.scatter(rows.target_raw_cycles, rows.error, s=7, alpha=0.35)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title=subset, xlabel="True RUL (cycles)", ylabel="Prediction − truth (cycles)")
fig.savefig(output / "test_residuals.png", dpi=160)
